In [ ]:
import os
import random
from pathlib import Path
from time import sleep
from typing import cast

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from tqdm import tqdm

In [2]:
ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / "data" / "fiqa"
INDEX_DIR = ROOT_DIR / "indexes"
DENSE_DIR = INDEX_DIR / "dense"


In [3]:
def create_client():
    load_dotenv()
    os.environ["NVIDIA_API_KEY"] = cast(str, (os.getenv("NVIDIA_API_KEY")))
    client = NVIDIAEmbeddings(model="nvidia/nemotron-3-embed-1b", trucate="NONE")

    return client

In [6]:
text_to_embed = """This dude built 3,000 landing pages across 1,000 U.S. cities to generate leads through SEO.

He didn’t wait around for results either. Within a few weeks the pages were already ranking on their own"""

In [ ]:
def load_corpus():
    return pd.read_parquet(DATA_DIR / "corpus.parquet")

In [ ]:
corpus = load_corpus()
len(corpus)

57638

In [7]:
embeddings = np.load(DENSE_DIR / "embeddings.npy", mmap_mode="r")
n_docs = len(corpus["_id"])
row_norms = np.empty(n_docs, dtype=np.float32)
chunk_size = 256
for start in range(0, n_docs, chunk_size):
    block = embeddings[start : (start + chunk_size)]
    row_norms[start : (start + chunk_size)] = np.linalg.norm(block, axis=1)




In [ ]:
Query="Where should I park my rainy-day funds?"
from utils.retrievers import DenseRetriever

dense=DenseRetriever()
query_norm=dense.embed_query(Query)

